# 04 - Scoring New Cases and Presentation Dashboard Data

This notebook shows how to:
- score a new patient/provider scenario
- generate a flashy but explainable risk summary
- create charts and presentation tables

This is the notebook you can demo live.

In [ ]:

import os
import json
import joblib
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

pd.set_option("display.max_columns", 120)

df = pd.read_parquet("data/model_features.parquet")
clf = joblib.load("artifacts/risk_classifier.joblib")
reg = joblib.load("artifacts/exposure_regressor.joblib")
src = joblib.load("artifacts/source_predictor.joblib")
fi = pd.read_csv("artifacts/feature_importance.csv")

df.head(2)

In [ ]:

# Example scenario builder
# You can replace the values below during your demo.
sample_case = {
    "average_covered_charges": 42000,
    "average_total_payments": 16500,
    "Billed amount": 6800,
    "Medicare payment": 2400,
    "provider_state": "CA",
    "service_type": "outpatient surgery with imaging",
    "drg_text": "major joint replacement with surgery and pathology review",
    "is_er": 0,
    "is_imaging": 1,
    "is_outpatient": 1,
    "is_surgery": 1,
    "high_complexity_drg": 1,
}

In [ ]:

# Bring in aggregate defaults from the training data by state
state_defaults = (
    df.groupby("provider_state")[[
        "provider_avg_gap", "provider_gap_std", "provider_avg_ratio", "provider_record_count",
        "state_avg_gap", "state_avg_ratio", "state_median_payment", "state_record_count",
        "provider_risk_index", "state_risk_index", "service_risk_weight"
    ]]
    .median()
    .reset_index()
)

ca_defaults = state_defaults[state_defaults["provider_state"] == sample_case["provider_state"]]
if ca_defaults.empty:
    ca_defaults = state_defaults.median(numeric_only=True).to_frame().T
else:
    ca_defaults = ca_defaults.drop(columns=["provider_state"])

row = pd.DataFrame([sample_case])

# derive features like notebook 2
eps = 1e-6
row["charge_ratio_inpatient"] = row["average_covered_charges"] / (row["average_total_payments"] + eps)
row["payment_gap_inpatient"] = row["average_covered_charges"] - row["average_total_payments"]
row["charge_ratio_outpatient"] = row["Billed amount"] / (row["Medicare payment"] + eps)
row["payment_gap_outpatient"] = row["Billed amount"] - row["Medicare payment"]
row["blended_charge_ratio"] = row["charge_ratio_outpatient"]
row["blended_payment_gap"] = row["payment_gap_outpatient"]
row["log_avg_covered"] = np.log1p(row["average_covered_charges"])
row["log_avg_payments"] = np.log1p(row["average_total_payments"])
row["log_billed_amount"] = np.log1p(row["Billed amount"])
row["log_medicare_payment"] = np.log1p(row["Medicare payment"])
row["log_blended_gap"] = np.log1p(np.clip(row["blended_payment_gap"], 0, None))

row = pd.concat([row.reset_index(drop=True), ca_defaults.reset_index(drop=True)], axis=1)

# recompute service risk if you want scenario-specific behavior
row["service_risk_weight"] = (
    0.35 * row["is_er"] +
    0.25 * row["is_surgery"] +
    0.15 * row["is_imaging"] +
    0.10 * 1 +
    0.15 * row["high_complexity_drg"]
).clip(0, 1)

row

In [ ]:

model_features = [
    "average_covered_charges", "average_total_payments", "Billed amount", "Medicare payment",
    "charge_ratio_inpatient", "payment_gap_inpatient", "charge_ratio_outpatient", "payment_gap_outpatient",
    "blended_charge_ratio", "blended_payment_gap", "log_avg_covered", "log_avg_payments",
    "log_billed_amount", "log_medicare_payment", "log_blended_gap",
    "is_er", "is_imaging", "is_outpatient", "is_surgery", "high_complexity_drg",
    "provider_avg_gap", "provider_gap_std", "provider_avg_ratio", "provider_record_count",
    "state_avg_gap", "state_avg_ratio", "state_median_payment", "state_record_count",
    "provider_risk_index", "state_risk_index", "service_risk_weight",
    "provider_state", "service_type", "drg_text"
]

score_X = row[model_features].copy()
risk_probability = float(clf.predict_proba(score_X)[:, 1][0])
exposure_prediction = float(reg.predict(score_X)[0])
source_probs = src.predict_proba(score_X)

source_labels = ["anesthesia", "radiology", "pathology"]
source_scores = {
    label: float(prob[0][1] if isinstance(prob, list) else prob[:, 1][0])
    for label, prob in zip(source_labels, source_probs)
}

result = {
    "risk_probability": risk_probability,
    "risk_band": "High" if risk_probability >= 0.67 else ("Moderate" if risk_probability >= 0.34 else "Low"),
    "predicted_exposure_amount": round(exposure_prediction, 2),
    "source_scores": source_scores,
}
result

In [ ]:

# Presentation-friendly risk summary
def make_checklist(service_type: str, risk_probability: float, source_scores: dict) -> list[str]:
    actions = [
        "Ask whether any clinicians or groups bill separately from the facility.",
        "Request a written estimate that includes facility and professional billing components."
    ]
    if source_scores.get("anesthesia", 0) >= 0.4:
        actions.append("Confirm whether the anesthesia group is in-network.")
    if source_scores.get("radiology", 0) >= 0.4:
        actions.append("Ask whether imaging will be interpreted by an external radiology group.")
    if source_scores.get("pathology", 0) >= 0.4:
        actions.append("Ask which lab or pathology group processes specimens and whether it is in-network.")
    if "er" in service_type.lower():
        actions.append("For emergency visits, verify post-visit billing protections and keep all itemized bills.")
    if risk_probability >= 0.6:
        actions.append("Call the insurer before care and document the reference number for the network-status call.")
    return actions

checklist = make_checklist(sample_case["service_type"], risk_probability, source_scores)
checklist

In [ ]:

# Flashy visuals for presentation
plt.figure(figsize=(6, 4))
plt.bar(["Risk Score"], [risk_probability])
plt.ylim(0, 1)
plt.title("Predicted Surprise Billing Risk")
plt.ylabel("Probability")
plt.tight_layout()
plt.show()

In [ ]:

plt.figure(figsize=(7, 4))
plt.bar(source_scores.keys(), source_scores.values())
plt.ylim(0, 1)
plt.title("Predicted Source-of-Billing Risk")
plt.ylabel("Probability")
plt.tight_layout()
plt.show()

In [ ]:

# Dashboard-ready aggregates
state_dashboard = (
    df.groupby("provider_state")
      .agg(
          avg_risk=("proxy_surprise_bill_label", "mean"),
          avg_exposure=("proxy_exposure_amount", "mean"),
          avg_gap=("blended_payment_gap", "mean"),
          avg_ratio=("blended_charge_ratio", "mean"),
          records=("provider_state", "size")
      )
      .reset_index()
      .sort_values("avg_risk", ascending=False)
)

provider_dashboard = (
    df.groupby(["provider_id", "provider_name", "provider_state"])
      .agg(
          avg_gap=("blended_payment_gap", "mean"),
          avg_ratio=("blended_charge_ratio", "mean"),
          avg_proxy_risk=("proxy_surprise_bill_label", "mean"),
          avg_exposure=("proxy_exposure_amount", "mean"),
          records=("provider_id", "size")
      )
      .reset_index()
      .sort_values(["avg_proxy_risk", "avg_exposure"], ascending=False)
)

state_dashboard.to_csv("artifacts/state_dashboard.csv", index=False)
provider_dashboard.to_csv("artifacts/provider_dashboard.csv", index=False)

state_dashboard.head(10)

In [ ]:

# Optional: scenario simulation
scenarios = []
for service_type, is_er, is_imaging, is_outpatient, is_surgery in [
    ("ER with imaging", 1, 1, 1, 0),
    ("outpatient imaging", 0, 1, 1, 0),
    ("planned surgery", 0, 0, 0, 1),
]:
    sim = row.copy()
    sim["service_type"] = service_type
    sim["is_er"] = is_er
    sim["is_imaging"] = is_imaging
    sim["is_outpatient"] = is_outpatient
    sim["is_surgery"] = is_surgery
    sim["service_risk_weight"] = (
        0.35 * sim["is_er"] +
        0.25 * sim["is_surgery"] +
        0.15 * sim["is_imaging"] +
        0.10 * 1 +
        0.15 * sim["high_complexity_drg"]
    ).clip(0, 1)

    risk = float(clf.predict_proba(sim[model_features])[:, 1][0])
    exposure = float(reg.predict(sim[model_features])[0])

    scenarios.append({
        "scenario": service_type,
        "risk_probability": risk,
        "predicted_exposure": exposure
    })

scenario_df = pd.DataFrame(scenarios).sort_values("risk_probability", ascending=False)
scenario_df

In [ ]:

plt.figure(figsize=(8, 4))
plt.bar(scenario_df["scenario"], scenario_df["risk_probability"])
plt.ylim(0, 1)
plt.title("Scenario Simulation - Risk Comparison")
plt.ylabel("Risk Probability")
plt.tight_layout()
plt.show()